In [33]:
!pip install cantera
!pip install CoolProp
import cantera as ct
print(ct.__version__)

3.2.0


In [55]:
import os
import pandas as pd
from CoolProp.CoolProp import PropsSI

methane_mass_fraction = 0.79 # 메탄 79%
nitrogen_mass_fraction = 0.21 # 질소 21%
FLUID = f"Methane[{methane_mass_fraction}]&Nitrogen[{nitrogen_mass_fraction}]"

def C_to_K(T_C):
    # 섭씨 온도를 켈빈 온도로 변환
    return T_C + 273.15

def K_to_C(T_K):
    # 켈빈 온도를 섭씨 온도로 변환
    return T_K - 273.15

def bar_to_Pa(P_bar):
    # bar 단위를 파스칼(Pa)로 변환
    return P_bar * 1e5

def Pa_to_bar(P_Pa):
    # 파스칼(Pa) 단위를 bar로 변환
    return P_Pa / 1e5

# =====================================================
# 1. CoolProp 속성 도우미 함수들
# =====================================================

def h_TP(T, P, fluid=FLUID):
    # 온도(T)와 압력(P)으로 질량 엔탈피(Hmass) 계산
    try:
        return PropsSI("Hmass", "T", T, "P", P, fluid)
    except ValueError:
        # 오류 발생 시 약간의 온도 보정
        return PropsSI("Hmass", "T", T + 0.1, "P", P, fluid)

def s_TP(T, P, fluid=FLUID):
    # 온도(T)와 압력(P)으로 질량 엔트로피(Smass) 계산
    try:
        return PropsSI("Smass", "T", T, "P", P, fluid)
    except ValueError:
        # 오류 발생 시 약간의 온도 보정
        return PropsSI("Smass", "T", T + 0.1, "P", P, fluid)

def T_Ph(P, h, fluid=FLUID):
    # 압력(P)과 질량 엔탈피(h)로 온도(T) 계산
    return PropsSI("T", "P", P, "Hmass", h, fluid)

def h_PS(P, s, fluid=FLUID):
    # 압력(P)과 질량 엔트로피(s)로 질량 엔탈피(Hmass) 계산 (등엔트로피 과정)
    return PropsSI("Hmass", "P", P, "Smass", s, fluid)

def rho_TP(T, P, fluid=FLUID):
    # 온도(T)와 압력(P)으로 질량 밀도(Dmass) 계산
    try:
        return PropsSI("Dmass", "T", T, "P", P, fluid)
    except ValueError:
        # 오류 발생 시 약간의 온도 보정
        return PropsSI("Dmass", "T", T + 0.1, "P", P, fluid)

# =====================================================
# 2. 컴포넌트 모델
# =====================================================

def liquid_pump(m_dot, T_in, P_in, P_out, eta=0.75):
    # 액체 펌프 모델
    h_in = h_TP(T_in, P_in)
    rho = rho_TP(T_in, P_in)

    if rho < 100.0:
        # 펌프 입구 밀도가 너무 낮으면 오류 발생 (액체가 아닌 증기/이상 상태일 가능성)
        raise ValueError(
            f"액체 펌프 입구 밀도가 너무 낮습니다 ({rho:.3f} kg/m3). "
            "입구는 증기/이상 상태일 가능성이 높습니다. "
            "펌프 입구 압력을 높이거나 펌프 입구 온도를 낮추십시오."
        )

    # 펌프 일 (엔탈피 변화)
    dh = (P_out - P_in) / rho / eta
    h_out = h_in + dh
    T_out = T_Ph(P_out, h_out)
    W = m_dot * dh

    return {
        "T_out": T_out, # 출구 온도
        "P_out": P_out, # 출구 압력
        "h_out": h_out, # 출구 엔탈피
        "W": W,         # 펌프 동력
        "rho_in": rho,  # 입구 밀도
    }

def compressor(m_dot, T_in, P_in, P_out, eta=0.75):
    # 압축기 모델
    if m_dot <= 0:
        # 질량 유량이 없으면 압축기 동력은 0
        return {
            "T_out": T_in,
            "P_out": P_out,
            "h_out": h_TP(T_in, P_out),
            "W": 0.0,
        }

    h_in = h_TP(T_in, P_in)
    s_in = s_TP(T_in, P_in)
    # 등엔트로피 출구 엔탈피
    h_out_s = h_PS(P_out, s_in)
    # 실제 출구 엔탈피 (효율 반영)
    h_out = h_in + (h_out_s - h_in) / eta
    T_out = T_Ph(P_out, h_out)
    # 압축기 동력
    W = m_dot * (h_out - h_in)

    return {
        "T_out": T_out,
        "P_out": P_out,
        "h_out": h_out,
        "W": W,
    }

def heat_exchanger_effectiveness(
    m_cold,
    T_cold_in,
    P_cold,
    m_hot,
    T_hot_in,
    P_hot,
    effectiveness=0.60,
    min_delta_T=3.0,
):
    """
    일반적인 열교환기 유용도(effectiveness) 모델.
    뜨거운 스트림이 차가운 스트림에 열을 전달합니다.
    """
    if m_hot <= 0 or effectiveness <= 0:
        # 뜨거운 유량 또는 유용도가 0 이하면 열 전달 없음
        return {
            "Q": 0.0,
            "T_cold_out": T_cold_in,
            "h_cold_out": h_TP(T_cold_in, P_cold),
            "T_hot_out": T_hot_in,
            "h_hot_out": h_TP(T_hot_in, P_hot),
        }

    h_hot_in = h_TP(T_hot_in, P_hot)
    h_cold_in = h_TP(T_cold_in, P_cold)
    # 최소 접근 온도 차이를 고려한 뜨거운 스트림의 최소 온도
    T_hot_min = T_cold_in + min_delta_T
    h_hot_min = h_TP(T_hot_min, P_hot)

    # 최대 가능한 열량 (뜨거운 스트림 기준)
    Q_max_hot = m_hot * max(h_hot_in - h_hot_min, 0.0)
    # 실제 열 전달량 (유용도 반영)
    Q = effectiveness * Q_max_hot

    # 출구 엔탈피 계산
    h_hot_out = h_hot_in - Q / m_hot
    h_cold_out = h_cold_in + Q / m_cold

    # 출구 온도 계산
    T_hot_out = T_Ph(P_hot, h_hot_out)
    T_cold_out = T_Ph(P_cold, h_cold_out)

    return {
        "Q": Q,                             # 열 전달량
        "T_cold_out": T_cold_out,         # 차가운 스트림 출구 온도
        "h_cold_out": h_cold_out,         # 차가운 스트림 출구 엔탈피
        "T_hot_out": T_hot_out,           # 뜨거운 스트림 출구 온도
        "h_hot_out": h_hot_out,           # 뜨거운 스트림 출구 엔탈피
    }

def set_stream_temperature(m_dot, T_in, P, T_target):
    """
    히터/쿨러를 적용하여 동일 압력에서 스트림 온도를 설정합니다.
    양의 Q는 스트림에 열이 추가됨을 의미합니다.
    """
    h_in = h_TP(T_in, P)
    h_out = h_TP(T_target, P)
    # 필요한 열량
    Q = m_dot * (h_out - h_in)

    return {
        "Q": Q,
        "T_out": T_target,
        "P_out": P,
        "h_out": h_out,
    }


# =====================================================
# 3. 다단 HPC 모델
# =====================================================

def get_hpc_stage_pressures(P_in, P_out, n_stages):
    """
    균등 압력비 단계.
    경계 압력: [P0, P1, ..., Pn]을 반환합니다.
    """
    pr_stage = (P_out / P_in) ** (1.0 / n_stages)
    pressures = [P_in]
    for _ in range(n_stages):
        pressures.append(pressures[-1] * pr_stage)
    pressures[-1] = P_out # 마지막 압력을 P_out으로 강제 설정하여 부동 소수점 오차 방지
    return pressures


def multistage_hpc_no_intercooling(m_dot, T_in, P_in, P_out, eta=0.75, n_stages=3):
    # 중간 냉각 없는 다단 고압 압축기(HPC) 모델
    pressures = get_hpc_stage_pressures(P_in, P_out, n_stages)
    T = T_in
    W_total = 0.0
    stage_results = []

    for i in range(n_stages):
        T_stage_in = T
        comp = compressor(m_dot, T_stage_in, pressures[i], pressures[i + 1], eta)
        W_total += comp["W"]
        T = comp["T_out"]

        stage_results.append({
            "stage": i + 1,
            "P_in": pressures[i],
            "P_out": pressures[i + 1],
            "T_in": T_stage_in,
            "T_out": comp["T_out"],
            "W": comp["W"],
        })

    return {
        "T_out": T,
        "P_out": P_out,
        "W": W_total,
        "stages": stage_results,
    }


# =====================================================
# 4. 전체 공정 모델 헬퍼 함수
# =====================================================

def _calculate_lng_stream(params, m_lng, P_lng_tank, T_lng_tank, P_lng_high, T_lng_pump_out_initial, m_bog_hpc, T_bog_comp_out, P_bog_comp_out):
    # LNG 펌프
    pump = liquid_pump(
        m_dot=m_lng,
        T_in=T_lng_tank,
        P_in=P_lng_tank,
        P_out=P_lng_high,
        eta=params["pump_eta"],
    )

    T_lng_after_pump_calc = pump["T_out"]
    P_lng_after_pump = pump["P_out"]

    # 펌프 출구 초기 가열
    pump_out_heater = set_stream_temperature(
        m_dot=m_lng,
        T_in=T_lng_after_pump_calc,
        P=P_lng_after_pump,
        T_target=T_lng_pump_out_initial,
    )
    T_lng_initial_heated = pump_out_heater["T_out"]
    Q_lng_pump_out_heating = pump_out_heater["Q"]

    # BOG 냉각용 디슈퍼히터 (LNG 스트림 이용)
    desuperheater = heat_exchanger_effectiveness(
        m_cold=m_lng,
        T_cold_in=T_lng_initial_heated,
        P_cold=P_lng_after_pump,
        m_hot=m_bog_hpc,
        T_hot_in=T_bog_comp_out,
        P_hot=P_bog_comp_out,
        effectiveness=params["desuperheater_effectiveness"]
    )

    T_lng_after_desuperheater = desuperheater["T_cold_out"]
    T_bog_hpc_in_after_desuperheater = desuperheater["T_hot_out"]
    Q_desuperheater = desuperheater["Q"]

    return {
        "pump": pump,
        "T_lng_after_pump_calc": T_lng_after_pump_calc,
        "P_lng_after_pump": P_lng_after_pump,
        "Q_lng_pump_out_heating": Q_lng_pump_out_heating,
        "T_lng_after_desuperheater": T_lng_after_desuperheater,
        "T_bog_hpc_in_after_desuperheater": T_bog_hpc_in_after_desuperheater,
        "Q_desuperheater": Q_desuperheater,
    }

def _assemble_simplified_results(
    params,
    m_lng,
    m_bog_hpc,
    comp_bog_initial,
    lng_stream_results,
    hpc_results,
    T_bog_after_valve,
    P_bog_hpc_out_actual,
    T_bog_final_sendout,
    P_bog_final_sendout,
    Q_bog_sendout_heating
):
    # LNG 스트림 결과
    pump = lng_stream_results["pump"]
    T_lng_after_pump_calc = lng_stream_results["T_lng_after_pump_calc"]
    P_lng_after_pump = lng_stream_results["P_lng_after_pump"]
    Q_lng_pump_out_heating = lng_stream_results["Q_lng_pump_out_heating"]
    T_lng_after_desuperheater = lng_stream_results["T_lng_after_desuperheater"]
    T_bog_hpc_in_after_desuperheater = lng_stream_results["T_bog_hpc_in_after_desuperheater"]
    Q_desuperheater = lng_stream_results["Q_desuperheater"]

    # BOG 스트림 결과
    W_hpc = hpc_results["W"]

    # 총 전력 및 열량 계산
    total_power_mw = (pump["W"] + comp_bog_initial["W"] + W_hpc) / 1e6
    bog_sendout_heating_duty_mw = Q_bog_sendout_heating / 1e6
    pump_out_heating_duty_mw = Q_lng_pump_out_heating / 1e6
    desuperheater_duty_mw = Q_desuperheater / 1e6

    return {
        "BOG_TOTAL_MASS_FLOW": m_bog_hpc, # 이제 BOG_TOTAL_MASS_FLOW는 HPC로 가는 유량과 동일
        "BOG_HPC_MASS_FLOW": m_bog_hpc,
        "BOG_PRECOOLER_MASS_FLOW": 0.0, # 재액화 경로 제거

        "LNG_TANK_T_C": K_to_C(params["T_lng_tank"]), # LNG 탱크 온도 (섭씨)
        "LNG_TANK_P_BAR": Pa_to_bar(params["P_lng_tank"]), # LNG 탱크 압력 (bar)
        "LNG_PUMP_OUT_CALC_T_C": K_to_C(T_lng_after_pump_calc), # LNG 펌프 출구 계산된 온도 (섭씨)
        "LNG_PUMP_OUT_P_BAR": Pa_to_bar(P_lng_after_pump), # LNG 펌프 출구 압력 (bar)
        "LNG_PUMP_OUT_HEATED_T_C": K_to_C(T_lng_after_desuperheater), # LNG 펌프 출구 가열된 온도 (디슈퍼히터 후) (섭씨)
        "LNG_PUMP_OUT_HEATING_DUTY_MW": pump_out_heating_duty_mw, # LNG 펌프 출구 가열 열량 (MW)

        "BOG_TANK_T_C": K_to_C(params["T_bog_tank"]),     # BOG 탱크 온도 (섭씨)
        "BOG_TANK_P_BAR": Pa_to_bar(params["P_bog_tank"]), # BOG 탱크 압력 (bar)
        "BOG_COMP_OUT_T_C": K_to_C(comp_bog_initial["T_out"]),       # BOG 압축기 출구 온도 (섭씨)
        "BOG_COMP_OUT_P_BAR": Pa_to_bar(comp_bog_initial["P_out"]), # BOG 압축기 출구 압력 (bar)
        "BOG_HPC_IN_T_C": K_to_C(T_bog_hpc_in_after_desuperheater), # BOG HPC 입구 온도 (디슈퍼히터 후) (섭씨)
        "BOG_HPC_OUT_T_C": K_to_C(hpc_results["T_out"]),         # BOG HPC 출구 온도 (섭씨)
        "BOG_HPC_OUT_P_BAR": Pa_to_bar(P_bog_hpc_out_actual),    # BOG HPC 출구 압력 (bar)
        "BOG_AFTER_VALVE_T_C": K_to_C(T_bog_after_valve), # BOG 밸브 통과 후 온도 (섭씨)
        "BOG_AFTER_VALVE_P_BAR": Pa_to_bar(params["P_sendout"]),   # BOG 밸브 통과 후 압력 (bar)

        "BOG_FINAL_SENDOUT_T_C": K_to_C(T_bog_final_sendout),       # BOG 최종 송출 온도 (섭씨)
        "BOG_FINAL_SENDOUT_P_BAR": Pa_to_bar(P_bog_final_sendout), # BOG 최종 송출 압력 (bar)
        "BOG_FINAL_SENDOUT_MASS_FLOW": m_bog_hpc, # BOG 최종 송출 질량 유량

        "DESUPERHEATER_DUTY_MW": desuperheater_duty_mw,    # 디슈퍼히터 열량 (MW)
        "BOG_SENDOUT_HEATING_DUTY_MW": bog_sendout_heating_duty_mw, # BOG 송출 가열 열량 (MW)
        "LNG_PUMP_POWER_MW": pump["W"] / 1e6,              # LNG 펌프 동력 (MW)
        "BOG_COMP_POWER_MW": comp_bog_initial["W"] / 1e6,          # BOG 압축기 동력 (MW)
        "BOG_HPC_POWER_MW": W_hpc / 1e6,                   # BOG HPC 동력 (MW)
        "TOTAL_POWER_MW": total_power_mw,                  # 총 동력 (MW)

        "HPC_STAGE_RESULTS": hpc_results["stages"],             # HPC 단계별 결과
    }

def simulate_process(params):
    # 주요 매개변수 추출
    m_lng = params["m_lng"]
    m_bog_total = params["m_bog_total"]

    # BOG 질량 유량 분배: BOG는 1개만 검토하므로 전체 유량이 HPC로 감
    m_bog_hpc = m_bog_total
    m_bog_pc = 0.0 # 재액화 경로 없음

    # 명확성을 위한 주요 매개변수 추출
    P_lng_tank = params["P_lng_tank"]
    T_lng_tank = params["T_lng_tank"]
    P_lng_high = params["P_lng_high"]
    T_lng_pump_out_initial = params["T_lng_pump_out_heated"]

    P_bog_tank = params["P_bog_tank"]
    T_bog_tank = params["T_bog_tank"]
    P_bog_comp_out = params["P_bog_comp_out"]
    P_bog_hpc_out = params["P_bog_hpc_out"]
    P_sendout = params["P_sendout"]
    T_sendout = params["T_sendout"]

    # 1. BOG 저압 압축
    comp_bog_initial = compressor(
        m_dot=m_bog_total,
        T_in=T_bog_tank,
        P_in=P_bog_tank,
        P_out=P_bog_comp_out,
        eta=params["comp_eta"],
    )
    T_bog_comp_out_initial = comp_bog_initial["T_out"]
    P_bog_comp_out_initial = comp_bog_initial["P_out"]

    # 2. LNG 펌프, 초기 가열 및 디슈퍼히터 (LNG cold side, BOG hot side)
    # LNG는 디슈퍼히터 출구까지만 검토
    lng_stream_results = _calculate_lng_stream(
        params, m_lng, P_lng_tank, T_lng_tank, P_lng_high, T_lng_pump_out_initial,
        m_bog_hpc, T_bog_comp_out_initial, P_bog_comp_out_initial
    )

    # BOG의 디슈퍼히터 출구 온도 (LNG 스트림 계산 결과에서 가져옴)
    T_bog_hpc_in_after_desuperheater = lng_stream_results["T_bog_hpc_in_after_desuperheater"]

    # 3. BOG 고압 압축 (HPC)
    hpc_results = multistage_hpc_no_intercooling(
        m_dot=m_bog_hpc,
        T_in=T_bog_hpc_in_after_desuperheater,
        P_in=P_bog_comp_out, # HPC 입구 압력은 저압 압축기 출구 압력과 동일
        P_out=P_bog_hpc_out,
        eta=params["comp_eta"],
        n_stages=3,
    )
    W_hpc = hpc_results["W"]
    T_bog_hpc_out = hpc_results["T_out"]
    P_bog_hpc_out_actual = hpc_results["P_out"]

    # 4. BOG 밸브 통과 후 (스로틀링, 등엔탈피)
    h_bog_before_valve = h_TP(T_bog_hpc_out, P_bog_hpc_out_actual)
    T_bog_after_valve = T_Ph(P_sendout, h_bog_before_valve) # 등엔탈피 스로틀링

    # 5. BOG 최종 송출 온도 설정 (필요한 경우 히팅/쿨링)
    bog_sendout_heater = set_stream_temperature(
        m_dot=m_bog_hpc,
        T_in=T_bog_after_valve,
        P=P_sendout,
        T_target=T_sendout,
    )
    Q_bog_sendout_heating = bog_sendout_heater["Q"]
    T_bog_final_sendout = bog_sendout_heater["T_out"]
    P_bog_final_sendout = bog_sendout_heater["P_out"]


    # 6. 결과 취합
    return _assemble_simplified_results(
        params=params,
        m_lng=m_lng,
        m_bog_hpc=m_bog_hpc,
        comp_bog_initial=comp_bog_initial,
        lng_stream_results=lng_stream_results,
        hpc_results=hpc_results,
        T_bog_after_valve=T_bog_after_valve,
        P_bog_hpc_out_actual=P_bog_hpc_out_actual,
        T_bog_final_sendout=T_bog_final_sendout,
        P_bog_final_sendout=P_bog_final_sendout,
        Q_bog_sendout_heating=Q_bog_sendout_heating
    )


# =====================================================
# 새로운 공정의 입력 조건
# =====================================================

params = {
    "m_lng": 65.0,

    # 총 BOG 유량, BOG 압축기 후 분할
    "m_bog_total": 35.0,
    "bog_to_hpc_frac": 1.0, # BOG를 1개만 검토하므로 1.0으로 설정

    # LNG 저장 탱크 및 실제 펌프 입구 조건
    "T_lng_tank": C_to_K(-160.0),
    "P_lng_tank": bar_to_Pa(5.0),

    # LNG 2차 펌프 출구는 펌핑 후 가열된다고 가정합니다.
    # 이것은 LNG 스트림의 디슈퍼히터 이전의 목표 온도입니다.
    "T_lng_pump_out_heated": C_to_K(-130.0),

    # LNG 2차 펌프 출구 압력
    "P_lng_high": bar_to_Pa(80.0),

    "T_bog_tank": C_to_K(-100.0),
    "P_bog_tank": bar_to_Pa(1.2),
    "P_bog_comp_out": bar_to_Pa(10.0),
    "P_bog_hpc_out": bar_to_Pa(80.0),

    "T_sendout": C_to_K(15.0),
    "P_sendout": bar_to_Pa(70.0),

    "pump_eta": 0.75,
    "comp_eta": 0.75,
    "desuperheater_effectiveness": 0.70, # 디슈퍼히터 효율에 대한 새로운 매개변수

    # Case 1 (기본) 별도 재액화 반환 조건. 이 매개변수는 더 이상 사용되지 않습니다.
    # "T_bog_reliq_return_case1": C_to_K(-124.0),
}


In [57]:
import pandas as pd
from CoolProp.CoolProp import PropsSI

def get_phase_string(T_C, P_bar, fluid=FLUID):
    T_K = C_to_K(T_C)
    P_Pa = bar_to_Pa(P_bar)
    try:
        raw_phase_output = PropsSI('Phase', 'T', T_K, 'P', P_Pa, fluid)

        phase_name_str = None

        if isinstance(raw_phase_output, (int, float)): # 정수 또는 실수 형태의 위상 코드를 처리
            phase_code = int(raw_phase_output) # 정수형으로 변환하여 매핑
            # CoolProp의 표준 위상 이름 문자열로 매핑합니다.
            if phase_code == 0:
                phase_name_str = 'undefined'
            elif phase_code == 1:
                phase_name_str = 'liquid'
            elif phase_code == 2:
                phase_name_str = 'gas'
            elif phase_code == 3:
                phase_name_str = 'supercritical'
            elif phase_code == 4:
                phase_name_str = 'supercritical_gas'
            elif phase_code == 5:
                phase_name_str = 'supercritical_liquid'
            elif phase_code == 6:
                phase_name_str = 'critical_point'
            elif phase_code == 7:
                phase_name_str = 'two_phase'
            else:
                phase_name_str = f'알 수 없는 위상 코드: {phase_code}'
        elif isinstance(raw_phase_output, str):
            phase_name_str = raw_phase_output
        else:
            # 예측하지 못한 다른 타입의 출력인 경우
            phase_name_str = '알 수 없음 (CoolProp 출력 형식 오류 - 타입 미확인)'

        # 매핑된 문자열 위상 이름을 한국어로 번역
        if phase_name_str == 'liquid':
            return '액체'
        elif phase_name_str == 'gas':
            return '기체'
        elif phase_name_str in ['supercritical', 'supercritical_gas', 'supercritical_liquid']:
            return '초임계' # 초임계 관련 위상들을 하나로 묶어 처리
        elif phase_name_str == 'critical_point':
            return '임계점'
        elif phase_name_str == 'two_phase':
            Q = PropsSI('Q', 'T', T_K, 'P', P_Pa, fluid)
            return f'기액 이중상 (건도: {Q:.2f})'
        elif phase_name_str == 'undefined':
            # 'undefined'인 경우, 가능하면 건도를 통해 추가 추정
            try:
                Q = PropsSI('Q', 'T', T_K, 'P', P_Pa, fluid)
                if Q is not None:
                    if Q <= 0: return '과냉각 액체 또는 포화 액체'
                    elif Q >= 1: return '과열 증기 또는 포화 증기'
                    else: return f'기액 이중상 (건도: {Q:.2f})'
            except ValueError:
                pass # Quality 계산 실패 시 무시
            return '정의되지 않음'
        else:
            return phase_name_str # CoolProp이 반환하는 기타 문자열 위상 이름

    except ValueError:
        # CoolProp 계산 오류 발생 시 (예: 정의되지 않은 영역), 건도를 통해 추가 추정
        try:
            Q = PropsSI('Q', 'T', T_K, 'P', P_Pa, fluid)
            if Q is not None:
                if Q <= 0: return '과냉각 액체 또는 포화 액체'
                elif Q >= 1: return '과열 증기 또는 포화 증기'
                else: return f'기액 이중상 (건도: {Q:.2f})'
        except Exception:
            pass # Quality 계산 실패 시 무시
        return '알 수 없음 (CoolProp 오류)'
    except Exception as e:
        return f'오류 발생: {type(e).__name__}'

# simulate_process 함수를 호출하여 results를 업데이트합니다.
results = simulate_process(params)

# 데이터 추출 및 가공
data_points = []

# --- LNG 스트림 --- (디슈퍼히터 출구까지만 고려)
# LNG 탱크
data_points.append({
    "위치": "LNG 탱크",
    "온도 [°C]": f"{results["LNG_TANK_T_C"]:.2f} (입력)",
    "압력 [bar]": f"{results["LNG_TANK_P_BAR"]:.2f} (입력)",
    "질량 유량 [kg/s]": f"{params["m_lng"]:.2f} (입력)",
    "위상": get_phase_string(results["LNG_TANK_T_C"], results["LNG_TANK_P_BAR"]),
    "다음 위치": "LNG 펌프 출구 (계산)"
})

# LNG 펌프 출구 (계산)
data_points.append({
    "위치": "LNG 펌프 출구 (계산)",
    "온도 [°C]": f"{results["LNG_PUMP_OUT_CALC_T_C"]:.2f} (계산)",
    "압력 [bar]": f"{results["LNG_PUMP_OUT_P_BAR"]:.2f} (입력)",
    "질량 유량 [kg/s]": f"{params["m_lng"]:.2f} (입력)",
    "위상": get_phase_string(results["LNG_PUMP_OUT_CALC_T_C"], results["LNG_PUMP_OUT_P_BAR"]),
    "다음 위치": "LNG 펌프 출구 (초기 가열 목표)"
})

# LNG 펌프 출구 (초기 가열 목표)
data_points.append({
    "위치": "LNG 펌프 출구 (초기 가열 목표)",
    "온도 [°C]": f"{K_to_C(params["T_lng_pump_out_heated"]):.2f} (입력)",
    "압력 [bar]": f"{results["LNG_PUMP_OUT_P_BAR"]:.2f} (입력)",
    "질량 유량 [kg/s]": f"{params["m_lng"]:.2f} (입력)",
    "위상": get_phase_string(K_to_C(params["T_lng_pump_out_heated"]), results["LNG_PUMP_OUT_P_BAR"]),
    "다음 위치": "LNG 디슈퍼히터 출구"
})

# LNG 디슈퍼히터 출구
data_points.append({
    "위치": "LNG 디슈퍼히터 출구",
    "온도 [°C]": f"{results["LNG_PUMP_OUT_HEATED_T_C"]:.2f} (계산)",
    "압력 [bar]": f"{results["LNG_PUMP_OUT_P_BAR"]:.2f} (입력)",
    "질량 유량 [kg/s]": f"{params["m_lng"]:.2f} (입력)",
    "위상": get_phase_string(results["LNG_PUMP_OUT_HEATED_T_C"], results["LNG_PUMP_OUT_P_BAR"]),
    "다음 위치": "(LNG 최종 스코프)"
})

# --- BOG 스트림 --- (단일 경로)
# BOG 탱크
data_points.append({
    "위치": "BOG 탱크",
    "온도 [°C]": f"{results["BOG_TANK_T_C"]:.2f} (입력)",
    "압력 [bar]": f"{results["BOG_TANK_P_BAR"]:.2f} (입력)",
    "질량 유량 [kg/s]": f"{params["m_bog_total"]:.2f} (입력)",
    "위상": get_phase_string(results["BOG_TANK_T_C"], results["BOG_TANK_P_BAR"]),
    "다음 위치": "BOG 저압 압축기 출구"
})

# BOG 저압 압축기 출구
data_points.append({
    "위치": "BOG 저압 압축기 출구",
    "온도 [°C]": f"{results["BOG_COMP_OUT_T_C"]:.2f} (계산)",
    "압력 [bar]": f"{results["BOG_COMP_OUT_P_BAR"]:.2f} (입력)",
    "질량 유량 [kg/s]": f"{params["m_bog_total"]:.2f} (입력)",
    "위상": get_phase_string(results["BOG_COMP_OUT_T_C"], results["BOG_COMP_OUT_P_BAR"]),
    "다음 위치": "BOG HPC 입구 (디슈퍼히터 출구)"
})

# BOG HPC 입구 (디슈퍼히터 출구)
data_points.append({
    "위치": "BOG HPC 입구 (디슈퍼히터 출구)",
    "온도 [°C]": f"{results["BOG_HPC_IN_T_C"]:.2f} (계산)",
    "압력 [bar]": f"{results["BOG_COMP_OUT_P_BAR"]:.2f} (입력)",
    "질량 유량 [kg/s]": f"{results["BOG_HPC_MASS_FLOW"]:.2f} (계산)",
    "위상": get_phase_string(results["BOG_HPC_IN_T_C"], results["BOG_COMP_OUT_P_BAR"]),
    "다음 위치": "BOG HPC 출구"
})

# BOG HPC 출구
data_points.append({
    "위치": "BOG HPC 출구",
    "온도 [°C]": f"{results["BOG_HPC_OUT_T_C"]:.2f} (계산)",
    "압력 [bar]": f"{results["BOG_HPC_OUT_P_BAR"]:.2f} (입력)",
    "질량 유량 [kg/s]": f"{results["BOG_HPC_MASS_FLOW"]:.2f} (계산)",
    "위상": get_phase_string(results["BOG_HPC_OUT_T_C"], results["BOG_HPC_OUT_P_BAR"]),
    "다음 위치": "BOG 밸브 통과 후"
})

# BOG 밸브 통과 후
data_points.append({
    "위치": "BOG 밸브 통과 후",
    "온도 [°C]": f"{results["BOG_AFTER_VALVE_T_C"]:.2f} (계산)",
    "압력 [bar]": f"{results["BOG_AFTER_VALVE_P_BAR"]:.2f} (입력)",
    "질량 유량 [kg/s]": f"{results["BOG_HPC_MASS_FLOW"]:.2f} (계산)",
    "위상": get_phase_string(results["BOG_AFTER_VALVE_T_C"], results["BOG_AFTER_VALVE_P_BAR"]),
    "다음 위치": "BOG 최종 송출"
})

# BOG 최종 송출
data_points.append({
    "위치": "BOG 최종 송출",
    "온도 [°C]": f"{results["BOG_FINAL_SENDOUT_T_C"]:.2f} (입력)",
    "압력 [bar]": f"{results["BOG_FINAL_SENDOUT_P_BAR"]:.2f} (입력)",
    "질량 유량 [kg/s]": f"{results["BOG_FINAL_SENDOUT_MASS_FLOW"]:.2f} (계산)",
    "위상": get_phase_string(results["BOG_FINAL_SENDOUT_T_C"], results["BOG_FINAL_SENDOUT_P_BAR"]),
    "다음 위치": "시스템 외부"
})

df_streams_full = pd.DataFrame(data_points)

print("\n=== 간소화된 시스템 스트림 데이터 요약 ===")
display(df_streams_full.round(2).fillna('N/A'))


=== 간소화된 시스템 스트림 데이터 요약 ===


,위치,온도 [°C],압력 [bar],질량 유량 [kg/s],위상,다음 위치
0,LNG 탱크,-160.00 (입력),5.00 (입력),65.00 (입력),임계점,LNG 펌프 출구 (계산)
1,LNG 펌프 출구 (계산),-131.49 (계산),80.00 (입력),65.00 (입력),과냉각 액체 또는 포화 액체,LNG 펌프 출구 (초기 가열 목표)
2,LNG 펌프 출구 (초기 가열 목표),-130.00 (입력),80.00 (입력),65.00 (입력),과냉각 액체 또는 포화 액체,LNG 디슈퍼히터 출구
3,LNG 디슈퍼히터 출구,-93.72 (계산),80.00 (입력),65.00 (입력),과냉각 액체 또는 포화 액체,(LNG 최종 스코프)
4,BOG 탱크,-100.00 (입력),1.20 (입력),35.00 (입력),초임계,BOG 저압 압축기 출구
5,BOG 저압 압축기 출구,61.28 (계산),10.00 (입력),35.00 (입력),초임계,BOG HPC 입구 (디슈퍼히터 출구)
6,BOG HPC 입구 (디슈퍼히터 출구),-73.59 (계산),10.00 (입력),35.00 (계산),초임계,BOG HPC 출구
7,BOG HPC 출구,113.59 (계산),80.00 (입력),35.00 (계산),초임계,BOG 밸브 통과 후
8,BOG 밸브 통과 후,111.83 (계산),70.00 (입력),35.00 (계산),초임계,BOG 최종 송출
9,BOG 최종 송출,15.00 (입력),70.00 (입력),35.00 (계산),초임계,시스템 외부
